# Streaming Ingestion Pipeline with Python and AWS Kinesis

In [26]:
import subprocess
from IPython.display import HTML
import boto3

### Creating both USA and International Kinesis Data Streams

In [27]:
USA_DATA_STREAM = "kinesis-project-usa-563649768086-data-stream"
INTERNATIONAL_DATA_STREAM = "kinesis-project-international-563649768086-data-stream"

def create_kinesis_data_stream(stream_name: str, shard_count: int) -> dict:
    """
    Create a Kinesis data stream with the specified name and shard count.
    
    Parameters:
    stream_name (str): The name of the Kinesis data stream to create.
    shard_count (int): The number of shards for the data stream.
    
    Returns:
    dict: A dictionary containing the response from the AWS CLI command.
    """
    client = boto3.client('kinesis')

    if stream_name in client.list_streams()['StreamNames']:
        print(f"Kinesis data stream '{stream_name}' already exists.")
        return 
    
    response = client.create_stream(StreamName=stream_name, ShardCount=shard_count)
    print("Kinesis data stream created successfully:", response)

create_kinesis_data_stream(stream_name=USA_DATA_STREAM, shard_count=2)
create_kinesis_data_stream(stream_name=INTERNATIONAL_DATA_STREAM, shard_count=2)

Kinesis data stream 'kinesis-project-usa-563649768086-data-stream' already exists.
Kinesis data stream 'kinesis-project-international-563649768086-data-stream' already exists.


In [28]:
def is_stream_active(stream_name: str) -> bool:
    """
    Check if a Kinesis data stream is active.
    
    Parameters:
    stream_name (str): The name of the Kinesis data stream to check.
    
    Returns:
    bool: True if the stream is active, False otherwise.
    """
    client = boto3.client('kinesis')
    response = client.describe_stream(StreamName=stream_name)
    return response['StreamDescription']['StreamStatus'] == 'ACTIVE'

print(is_stream_active(USA_DATA_STREAM))
print(is_stream_active(INTERNATIONAL_DATA_STREAM))

True
True


### Creating the Fireshouse Stream

#### The IAM Roles

In [29]:
def create_firehose_role(role_name: str, bucket_name: str, stream_arn: str, 
                          log_group_name: str, account_id: str, region: str) -> str:
    iam = boto3.client('iam')

    trust_policy = {
        "Version": "2012-10-17",
        "Statement": [
            {
                "Effect": "Allow",
                "Principal": {"Service": "firehose.amazonaws.com"},
                "Action": "sts:AssumeRole",
                "Condition": {
                    "StringEquals": {"sts:ExternalId": account_id}
                }
            }
        ]
    }

    try:
        role = iam.get_role(RoleName=role_name)
        print(f"Role '{role_name}' already exists.")
        return role['Role']['Arn']
    except iam.exceptions.NoSuchEntityException:
        pass

    role = iam.create_role(
        RoleName=role_name,
        AssumeRolePolicyDocument=json.dumps(trust_policy),
        Description="Role for Kinesis Firehose delivery stream"
    )

    permissions_policy = {
        "Version": "2012-10-17",
        "Statement": [
            {
                "Effect": "Allow",
                "Action": [
                    "s3:AbortMultipartUpload",
                    "s3:GetBucketLocation",
                    "s3:GetObject",
                    "s3:ListBucket",
                    "s3:ListBucketMultipartUploads",
                    "s3:PutObject"
                ],
                "Resource": [
                    f"arn:aws:s3:::{bucket_name}",
                    f"arn:aws:s3:::{bucket_name}/*"
                ]
            },
            {
                "Effect": "Allow",
                "Action": [
                    "kinesis:DescribeStream",
                    "kinesis:GetShardIterator",
                    "kinesis:GetRecords",
                    "kinesis:ListShards"
                ],
                "Resource": stream_arn
            },
            {
                "Effect": "Allow",
                "Action": ["logs:PutLogEvents"],
                "Resource": f"arn:aws:logs:{region}:{account_id}:log-group:{log_group_name}:*"
            }
        ]
    }

    iam.put_role_policy(
        RoleName=role_name,
        PolicyName=f"{role_name}-policy",
        PolicyDocument=json.dumps(permissions_policy)
    )

    print(f"Role '{role_name}' created with attached policy.")
    return role['Role']['Arn']

In [30]:
import time
import json

create_firehose_role(
    role_name="kinesis-project-usa-563649768086-firehose-role",
    bucket_name="kinesis-project-usa-563649768086",
    stream_arn=f"arn:aws:kinesis:us-east-1:563649768086:stream/{USA_DATA_STREAM}",
    log_group_name="/aws/kinesisfirehose/kinesis-project-usa-563649768086-firehose",
    account_id="563649768086",
    region="us-east-1"
)

create_firehose_role(
    role_name="kinesis-project-international-563649768086-firehose-role",
    bucket_name="kinesis-project-international-563649768086",
    stream_arn=f"arn:aws:kinesis:us-east-1:563649768086:stream/{INTERNATIONAL_DATA_STREAM}",
    log_group_name="/aws/kinesisfirehose/kinesis-project-international-563649768086-firehose",
    account_id="563649768086",
    region="us-east-1"
)

# IAM roles take a few seconds to propagate — give it a buffer before Firehose tries to assume it
time.sleep(15)


Role 'kinesis-project-usa-563649768086-firehose-role' already exists.
Role 'kinesis-project-international-563649768086-firehose-role' already exists.


#### The Firehose Stream

In [ ]:
def create_kinesis_firehose_delivery_stream(stream_name: str, firehose_name: str, bucket_name: str, role_name: str, log_group_name: str, log_stream_name: str, account_id: str, region: str) -> dict:
    """
    Create a Kinesis Firehose delivery stream with the specified name and S3 destination.
    
    Parameters:
    stream_name (str): The name of the Kinesis Firehose delivery stream to create.
    firehose_name (str): The name of the Firehose delivery stream.
    bucket_name (str): The name of the S3 bucket where data will be delivered.
    role_name (str): The name of the IAM role for the Firehose delivery stream.
    log_group_name (str): The name of the CloudWatch log group.
    log_stream_name (str): The name of the CloudWatch log stream.
    account_id (str): The AWS account ID.
    region (str): The AWS region.

    Returns:
    dict: A dictionary containing the response from the AWS CLI command.
    """
    client = boto3.client('firehose')

    if firehose_name in client.list_delivery_streams()['DeliveryStreamNames']:
        print(f"Kinesis Firehose delivery stream '{firehose_name}' already exists.")
        return 
    
    response = client.create_delivery_stream(
        DeliveryStreamName=firehose_name,
        DeliveryStreamType='KinesisStreamAsSource',
        S3DestinationConfiguration={
            'RoleARN': f'arn:aws:iam::{account_id}:role/{role_name}',
            'BucketARN': f'arn:aws:s3:::{bucket_name}',
            'Prefix': "firehose-data/",
            'ErrorOutputPrefix': f'{stream_name}/error/',
            'BufferingHints': {
                'SizeInMBs': 1,
                'IntervalInSeconds': 60
            },
            'CompressionFormat': 'UNCOMPRESSED',
            'CloudWatchLoggingOptions': {
                'Enabled': True,
                'LogGroupName': log_group_name,
                'LogStreamName': log_stream_name
            }
        },
        KinesisStreamSourceConfiguration={
            'KinesisStreamARN': f'arn:aws:kinesis:{region}:{account_id}:stream/{stream_name}',
            'RoleARN': f'arn:aws:iam::{account_id}:role/{role_name}'
        }
    )
    print("Kinesis Firehose delivery stream created successfully:", response)


create_kinesis_firehose_delivery_stream(
    stream_name=USA_DATA_STREAM,
    firehose_name="kinesis-project-usa-563649768086-firehose",
    bucket_name="kinesis-project-usa-563649768086",
    role_name="kinesis-project-usa-563649768086-firehose-role",
    log_group_name="/aws/kinesisfirehose/kinesis-project-usa-563649768086-firehose",
    log_stream_name="kinesis-project-usa-563649768086-firehose-log-stream",
    account_id="563649768086",  
    region="us-east-1"
)

create_kinesis_firehose_delivery_stream(
    stream_name=INTERNATIONAL_DATA_STREAM,
    firehose_name="kinesis-project-international-563649768086-firehose",
    bucket_name="kinesis-project-international-563649768086",
    role_name="kinesis-project-international-563649768086-firehose-role",
    log_group_name="/aws/kinesisfirehose/kinesis-project-international-563649768086-firehose",
    log_stream_name="kinesis-project-international-563649768086-firehose-log-stream",
    account_id="563649768086",
    region="us-east-1"
)

Kinesis Firehose delivery stream 'kinesis-project-usa-563649768086-firehose' already exists.
Kinesis Firehose delivery stream 'kinesis-project-international-563649768086-firehose' already exists.


: 